# L6: Multi-agent Collaboration for Financial Analysis



In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [9]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [10]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [11]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [12]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [13]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [29]:
# Example data for kicking off the process

sanofi_inputs = {
    'stock_selection': 'Sanofi',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Long Term Investment',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [30]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs= sanofi_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (Sanofi). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
I need to delegate the task of continuously monitoring and analyzing market data for Sanofi to the Data Analyst, as they have the expertise in statistical modeling and machine learning required for this task.

Action: Delegate work to co-worker
Action Input: {"coworker": "Data Analyst", "task": "Continuously monitor and analyze market data for Sanofi", "context": "The Data Analyst is tasked with using statistical modeling and machine learning to identify trends and predict market movements for Sanofi. This involves analyzing historical data, identifying patterns, and providing insights and alerts about significant market opportunities or threats."}


> Entering new CrewAgentExecutor chain...
I need to gather rea

I should start by gathering relevant information on Sanofi's market trends through statistical methods and machine learning techniques.

Action: Read website content
Action Input: {"website_url": "https://www.sanofi.com/en"} 

R&D-Driven and AI-Powered Biopharmaceutical Company | SanofiSanofiOur CompanyOur CompanyOur StrategyScience with PurposeGovernanceGovernanceBoard of DirectorsExecutive CommitteeSpecialist CommitteesSocial ImpactSocial ImpactSustainable and Equitable Access to Healthcare Environmental ImpactHealth Systems’ ResilienceCode of Conduct and Responsible BusinessInclusionGlobal Health UnitOur LegacyOur ScienceOur ScienceOur PipelineTherapeutic AreasTherapeutic AreasImmunologyNeurologyOncologyRare DiseasesVaccinesTechnology Platforms Research PublicationsClinical Trials & ResultsClinical Trials & ResultsFind a Clinical TrialClinical Study ResultsInvestigator Sponsored StudiesDiversity in Clinical TrialsExternal InnovationExternal InnovationExternal Innovation in North Ame

Action: Search the internet
Action Input: {"search_query": "Sanofi stock market trends and predictions"} 


Search results: Title: Sanofi (SNY)
Link: https://finance.yahoo.com/quote/SNY/
Snippet: The major indices are lower at midday on Thursday. Crude oil continues to climb again and is at $86 per barrel.
---
Title: Share Price: Performance and Insights
Link: https://www.sanofi.com/en/investors/sanofi-share-and-adrs/stock-chart
Snippet: Sanofi stock performance with up-to-date charts and analysis. Understand the value and prospects of investing in Sanofi company stock.
---
Title: Sanofi (SASY) Stock Forecast & Price Target
Link: https://www.investing.com/equities/sanofi-aventis-consensus-estimates
Snippet: According to projections from 23 analysts, the average 12-month price target for Sanofi is 94.28 EUR, with a high estimate of 112 EUR and a low estimate of 78 ...
---
Title: SNY Stock Quote Price and Forecast
Link: https://www.cnn.com/markets/stocks/SNY
Snippet: The price of SNY sha

APIError: The server had an error while processing your request. Sorry about that!

- Display the final result as Markdown.

In [31]:
from IPython.display import Markdown
Markdown(result)

Risk Analysis Report for Sanofi Trading Strategies:

After a thorough analysis of the market opportunities and threats for Sanofi, it is evident that there are significant growth opportunities for the company in the pharmaceutical market. Sanofi is an R&D driven and AI-powered biopharmaceutical company committed to improving people's lives and delivering compelling growth. They focus on breakthrough innovations in medicines and vaccines, with a strong pipeline and a commitment to sustainability. Sanofi has a diverse portfolio across various therapeutic areas and is actively involved in clinical trials and external innovation. They have a strong focus on patient engagement and understanding diseases and conditions. The company has a history of successful financial performance and recent approvals for new products.

In terms of execution methods for trades on Sanofi given the current market conditions and stock price, it is recommended to implement a combination of market monitoring, stop-loss orders, and leveraging options or derivatives to optimize profitability while mitigating risks effectively. Additionally, regular monitoring of market trends, clinical trial results, and regulatory approvals is crucial for timely and informed decision-making.

To ensure that the trading strategies developed align with the user's risk tolerance, it is important to incorporate stop-loss orders, diversification, and position sizing into the strategies. By diversifying the portfolio and setting clear risk management strategies, the user's risk tolerance can be effectively managed while maximizing returns.

In conclusion, the detailed execution plans for trading Sanofi should focus on leveraging market opportunities, implementing risk management strategies, and aligning with the user's risk tolerance to optimize profitability and mitigate risks effectively. Regular monitoring of market trends, clinical trial results, and regulatory approvals is crucial for informed decision-making. Incorporating clear risk management strategies and position sizing into the trading plans can help manage risk effectively while maximizing returns.